In [1]:
import os
import pickle 

In [2]:
def extract_location_model_cascade(text):
    # Initialize parsed results
    gpt4_choice = None
    assistant_location = None
    
    # Split and extract the 'GPT4 Assistant' choice
    if 'GPT4 Assistant:\n    Choice:' in text:
        # Get the text part after 'GPT4 Assistant:\n    Choice:'
        gpt4_part = text.split('GPT4 Assistant:\n    Choice:')[1]
        gpt4_choice = gpt4_part.split('\n', 1)[0].strip()
        gpt4_choice = gpt4_choice.lower()

    # Split and extract the 'Assistant' location
    if 'Location:\n     [/INST]' in text:
        assistant_part = text.split('Location:\n     [/INST]')[1]
        assistant_location = assistant_part.strip()
        assistant_location = assistant_location.lower()

    return gpt4_choice, assistant_location

In [3]:
# for no-cascade case 
def extract_location(text):
    # Convert the text to lowercase to ensure case-insensitive matching
    text = text.lower()
    
    text = text.split('location:', 1)[1]
    # List of possible choices
    choices = ['top left', 'top right', 'bottom left', 'bottom right', 'none']
    
    # Check each choice in the text and return the first match
    for choice in choices:
        if choice in text:
            return choice
    # Return None if no match is found
    return 'none'

def extract_location_flamingo(text):
    # Convert the text to lowercase to ensure case-insensitive matching
    text = text.lower()
    
    # List of possible choices
    choices = ['top left', 'top right', 'bottom left', 'bottom right', 'none']
    
    # Check each choice in the text and return the first match
    for choice in choices:
        if choice in text:
            return choice
    # Return None if no match is found
    return 'none'

def count_error(test_location, gt_location):
    gt_location = gt_location.lower()
    if gt_location == test_location:
        return 0
    return 1

def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
        return data
    

In [4]:
# need to modify
out_dir = '/scratch/bbyr/mw34/UOUO/output8'
file_names = os.listdir(out_dir)
file_paths = [os.path.join(out_dir, f) for f in file_names]

In [34]:
# acc for small vlm w.r.t. big-vlm(gt) answer (not in use now)
total = 0
error = 0
for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        for sub_key, sub_value in value.items():
            # Parse the location from the text
            test_parsed_location = extract_location(sub_value[0])
            error += count_error(test_parsed_location, sub_value[1])
            total += 1

print((total-error)/ total)
        

IndexError: list index out of range

In [ ]:
# acc for small vlm w.r.t. gpt-4o answer(in use)
total = 0
error = 0
for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        for sub_key, sub_value in value.items():
            # Parse the location from the text
            print(sub_value)
            gt_parsed_location,test_parsed_location = extract_location_model_cascade(sub_value[0])
            # print(gt_parsed_location, test_parsed_location)
            error += count_error(test_parsed_location, gt_parsed_location)
            total += 1

print((total-error)/ total)

In [18]:
# shuffle

251

In [6]:
with open('/scratch/bczf/zoezheng126/uouo/mosaic/preprocess/category_lookup.pkl', 'rb') as f:
    file_dict = pickle.load(f)

In [7]:
def get_mosaic_groundtruth(mosaic_filename, file_dict):
    mosaic_id = mosaic_filename.split('/')[-1]
    four_file_names = mosaic_id.split('_')[1:]
    four_cat_names = [file_dict[file] for file in four_file_names]
    ans_dict = dict()
    idx_dict = {0:"top left", 1: "top right", 2: "bottom left", 3: "bottom right"}
    for i in range(len(four_file_names)):
        ans_dict[four_cat_names[i].lower()] = idx_dict[i]
    return ans_dict

def get_gpt_location(text):
    locations = ['top left', 'top right', 'bottom left', 'bottom right', 'none']
    for l in locations: 
        if l in text.lower():
            return l

In [ ]:
# shuffle and model cascade from gpt-4o, acc for small vlm w.r.t. gt answer
total = 0
error = 0
for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        ans_dict = get_mosaic_groundtruth(key, file_dict)
        print(value)
        for sub_key, sub_value in value.items():
            try:
                # test_parsed_location = extract_location(sub_value[0])
                # print(sub_value)
                gt_parsed_location,test_parsed_location = extract_location_model_cascade(sub_value[0])
                gt_cat_location = ans_dict[sub_key.lower()]
                # if test_parsed_location != gt_cat_location: 
                #     print(test_parsed_location, gt_parsed_location, gt_cat_location)
                # print(sub_value[0])
                # print('\n')
                error += count_error(test_parsed_location, gt_cat_location)
                total += 1
            except: 
                print(sub_key)
                continue 

print((total-error)/ total)

In [ ]:
# [Llava]shuffle and no model cascade, acc for small vlm w.r.t. gt answer
total = 0
error = 0
for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        ans_dict = get_mosaic_groundtruth(key, file_dict)
        for sub_key, sub_value in value.items():
            try: 
                test_parsed_location = extract_location(sub_value[0])
                gt_cat_location = ans_dict[sub_key.lower()]
                if test_parsed_location != gt_cat_location: 
                    print(test_parsed_location, gt_cat_location)
                # print('\n')
                error += count_error(test_parsed_location, gt_cat_location)
                total += 1
            except:
                continue

print((total-error)/ total)

In [9]:
# [Flamingo] shuffle and no model cascade, acc for small vlm w.r.t. gt answer
total = 0
error = 0
for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        ans_dict = get_mosaic_groundtruth(key, file_dict)
        for sub_key, sub_value in value.items():
            try: 
                test_parsed_location = extract_location_flamingo(sub_value[0])
                if 'none' in test_parsed_location:
                    continue
                gt_cat_location = ans_dict[sub_key.lower()]
                if test_parsed_location != gt_cat_location: 
                    print(test_parsed_location, gt_cat_location)
                # print('\n')
                error += count_error(test_parsed_location, gt_cat_location)
                total += 1
            except:
                continue

print((total-error)/ total)

bottom left top left
bottom right top right
bottom right top left
bottom left top right
top right bottom right
bottom right bottom left
bottom left top right
top left bottom right
bottom right top right
bottom right top left
bottom left top right
bottom left top left
top right bottom left
bottom right top left
bottom left top right
top left bottom left
top right top left
bottom left top right
top right bottom left
bottom right top left
top left bottom right
top left top right
bottom right top right
bottom left top right
top left bottom left
top left bottom left
bottom right top right
bottom left top left
top left top right
bottom right top right
bottom right bottom left
top right bottom right
bottom left bottom right
bottom right bottom left
top right top left
top left top right
bottom right top right
top right bottom left
bottom right bottom left
top right bottom left
bottom right bottom left
bottom left top left
bottom right top left
bottom left top left
top right bottom right
bottom

In [ ]:
# gpt-4o answer extraction
total = 0
error = 0
gpt_error = 0
gpt_total = 0
num_of_mosaic_count = 0
for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        ans_dict = get_mosaic_groundtruth(key, file_dict)
        num_of_mosaic_count += 1
        for subkey, subvalue in value.items():
            # print(value)
            try: 
                message = subvalue['choices'][0]['message']['content']
            except: 
                continue 
            gpt_location = get_gpt_location(message)
            gt_location = ans_dict[subkey.lower()].lower()
            if gpt_location != gt_location:
                error += 1
            total += 1
        # print('\n')

print((total-error)/ total)

In [ ]:
# gpt-4o answer extraction and save to a pkl
total = 0
error = 0
new_data = {}  # Create a new dictionary to store the extracted data

for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        ans_dict = get_mosaic_groundtruth(key, file_dict)
        new_data[key] = {}  # Initialize a nested dictionary for each key

        for subkey, subvalue in value.items():
            try: 
                message = subvalue['choices'][0]['message']['content']
            except: 
                continue 

            # Finding the location after 'Location:'
            if 'Location:' in message:
                location = message.split('Location:')[-1].strip()  # Extracts and strips extra spaces
                print(f"{subkey}: {location}")

                new_data[key][subkey.lower()] = location.strip().lower()
                
                if ans_dict[subkey.lower()].lower() != location.strip().lower():
                    error += 1
            elif 'none' in message:
                new_data[key][subkey.lower()] = 'none'
                error += 1
            
            total += 1

# Write the new data dictionary to a pickle file
with open('gpt-4o-mosaic-shuffle-2-ans.pkl', 'wb') as f:
    pickle.dump(new_data, f)

print((total - error) / total)


In [32]:
# gpt-4o answer (with reasoning) extraction and save to a pkl
total = 0
error = 0
new_data = {}  # Create a new dictionary to store the extracted data

for file_path in file_paths:
    data = load_pickle(file_path)
    for key, value in data.items():
        ans_dict = get_mosaic_groundtruth(key, file_dict)
        new_data[key] = {}  # Initialize a nested dictionary for each key

        for subkey, subvalue in value.items():
            try: 
                message = subvalue['choices'][0]['message']['content']
                # print(message)
                # print('\n')
            except: 
                print(message)
                continue 

            # Finding the location after 'Location:'
            
            if 'Location:' in message:
                after_location = message.split('Location:')[-1]  # Extracts and strips extra spaces
                if 'Reason' in after_location:
                    location, reason = after_location.split('Reason:')
                else: 
                    location = after_location

                # print(f"{subkey}: {location}")
                # print(f'reason: {reason}')

                new_data[key][subkey.lower()] = dict()
                new_data[key][subkey.lower()]['big_vlm_loc_ans'] = location.strip().lower()
                new_data[key][subkey.lower()]['reason'] = reason.strip().lower()
                
                if ans_dict[subkey.lower()].lower() != location.strip().lower():
                    error += 1
            else: 
                print(message)
                new_data[key][subkey.lower()] = dict()
                new_data[key][subkey.lower()]['big_vlm_loc_ans'] = 'bottom right'
                new_data[key][subkey.lower()]['reason'] = 'The object in the bottom right section of the mosaic image is a device that appears to be a bead blaster based on its design and components such as the nozzle and pressure gauge.'
            
            total += 1

# Write the new data dictionary to a pickle file
with open('gpt-4o-mosaic-four-mmd-reason-ans.pkl', 'wb') as f:
    pickle.dump(new_data, f)

print((total - error) / total)


**object name**: Bead blaster  
**Location**: bottom right  
**Reason**: The object in the bottom right section of the mosaic image is a device that appears to be a bead blaster based on its design and components such as the nozzle and pressure gauge.
0.7747326203208557
